In [22]:
from data import *
import numpy as np
filestream = open(r'C:\Users\ProfDrJannikHüls\OneDrive - Richters & Hüls Ingenieurbüro für Abfallwirtschaft und Immissionsschutz\Desktop\NorsonicMessung.xlsm', 'r')
df = pd.read_excel(r'C:\Users\ProfDrJannikHüls\OneDrive - Richters & Hüls Ingenieurbüro für Abfallwirtschaft und Immissionsschutz\Desktop\NorsonicMessung.xlsm', 
sheet_name='Profile', 
skiprows=1, 
usecols='A:E',
names=['Zeitstempel', 'LAeq', 'LAFmax', 'LCeq','Markers'])
df['Markers'] = df['Markers'].str.replace('Battery;', '', regex=False)
df['Markers'] = df['Markers'].str.replace('Stop;', '', regex=False)

intervals = []
in_interval = False
start_time = None

for idx, row in df.iterrows():
    marker = row['Markers'].strip()
    if marker != '':
        if not in_interval:
            start_time = row['Zeitstempel']
            interval_marker = marker.replace(';', '')
            in_interval = True
        end_time = row['Zeitstempel']
    else:
        if in_interval:
            intervals.append((start_time, end_time, interval_marker))
            in_interval = False

# If the last row is part of an interval, close it
if in_interval:
    intervals.append((start_time, end_time, interval_marker))

# Compute logarithmic mean LAeq for each interval
log_means = []
for start, end, marker in intervals:
    mask = (df['Zeitstempel'] >= start) & (df['Zeitstempel'] <= end)
    values = df.loc[mask, 'LAeq'].values
    if len(values) > 0:
        # LAeq is in dB, so convert to linear, mean, then back to dB
        linear = 10 ** (values / 10)
        log_mean = 10 * np.log10(linear.mean())
        log_means.append((start, end, marker, log_mean))
    else:
        log_means.append((start, end, marker, np.nan))

log_means

intervals

[(' 10.04.2025 04:45:41,672', ' 10.04.2025 04:47:41,672', 'Fremd'),
 (' 10.04.2025 04:50:39,672', ' 10.04.2025 04:53:03,672', 'Fremd'),
 (' 10.04.2025 04:56:58,672', ' 10.04.2025 04:57:31,672', 'Fremd'),
 (' 10.04.2025 04:57:58,672', ' 10.04.2025 04:59:12,672', 'Fremd'),
 (' 10.04.2025 05:04:51,672', ' 10.04.2025 05:05:45,672', 'Fremd'),
 (' 10.04.2025 05:08:26,672', ' 10.04.2025 05:15:47,672', 'Silo')]

In [30]:
# Create a DataFrame for intervals with logarithmic means for LAeq, LAFmax, and LCeq
markers_data = []
intervals = []
in_interval = False
start_time = None

global_starttime = df['Zeitstempel'].min()
global_endtime = df['Zeitstempel'].max()

intervals.append((global_starttime, global_endtime, 'Gesamt'))
intervals.append((global_starttime, global_endtime, 'Ohne Marker'))

for idx, row in df.iterrows():
    marker = row['Markers'].strip()
    if marker != '':
        if not in_interval:
            start_time = row['Zeitstempel']
            interval_marker = marker.replace(';', '')
            in_interval = True
        end_time = row['Zeitstempel']
    else:
        if in_interval:
            intervals.append((start_time, end_time, interval_marker))
            in_interval = False

# If the last row is part of an interval, close it
if in_interval:
    intervals.append((start_time, end_time, interval_marker))

for start, end, marker in intervals:
    if marker != 'Ohne Marker':
        mask = (df['Zeitstempel'] >= start) & (df['Zeitstempel'] <= end)
    else:
        mask = (df['Zeitstempel'] >= start) & (df['Zeitstempel'] <= end) & (df['Markers'] == '')
    row = {'starttime': start, 'endtime': end, 'marker': marker}
    for col in ['LAeq', 'LAFmax', 'LCeq']:
        values = df.loc[mask, col].values
        if len(values) > 0:
            linear = 10 ** (values / 10)
            log_mean = 10 * np.log10(linear.mean())
        else:
            log_mean = np.nan
        row[f'{col}'] = log_mean
    markers_data.append(row)

markersdf = pd.DataFrame(markers_data)
markersdf

,starttime,endtime,marker,LAeq,LAFmax,LCeq
0,"10.04.2025 04:42:29,672","10.04.2025 05:15:47,672",Gesamt,43.680663,44.440906,58.734951
1,"10.04.2025 04:42:29,672","10.04.2025 05:15:47,672",Ohne Marker,41.708332,42.393618,56.905000
2,"10.04.2025 04:45:41,672","10.04.2025 04:47:41,672",Fremd,48.489944,49.266925,59.359994
3,"10.04.2025 04:50:39,672","10.04.2025 04:53:03,672",Fremd,47.296684,48.111554,59.961784
4,"10.04.2025 04:56:58,672","10.04.2025 04:57:31,672",Fremd,42.472237,43.495624,58.061701
5,"10.04.2025 04:57:58,672","10.04.2025 04:59:12,672",Fremd,43.006026,43.813803,59.999194
6,"10.04.2025 05:04:51,672","10.04.2025 05:05:45,672",Fremd,45.239153,45.841064,58.023768
7,"10.04.2025 05:08:26,672","10.04.2025 05:15:47,672",Silo,43.407683,44.241752,61.038871


In [61]:
file = r"C:\Users\ProfDrJannikHüls\OneDrive - Richters & Hüls Ingenieurbüro für Abfallwirtschaft und Immissionsschutz\Desktop\NorsonicMessung.xlsm"
spektren = pd.read_excel(
    file,
    sheet_name='Global',
    usecols='BA:CE',
    skiprows=1,
    nrows=1
).T
spektren.columns = ['dB(A)']
spektren.index.name = 'Frequenz'
spektren.index = spektren.index.str.replace('G3_FRQ_LEQ_', '', regex=False)
spektren

,dB(A)
Frequenz,
20Hz,51.0
25Hz,56.1
31.5Hz,48.6
40Hz,55.8
50Hz,52.5
63Hz,45.8
80Hz,47.9
100Hz,39.9
125Hz,33.6


In [62]:
summarkerdf = (
            markersdf.groupby('marker')[['LAeq', 'LCeq', 'LAFmax']]
            .apply(lambda df: 10 * np.log10((10 ** (df / 10)).mean()))
            .reset_index()
        )

In [46]:
markersdf

,starttime,endtime,marker,LAeq,LAFmax,LCeq
0,"10.04.2025 04:42:29,672","10.04.2025 05:15:47,672",Gesamt,43.680663,44.440906,58.734951
1,"10.04.2025 04:42:29,672","10.04.2025 05:15:47,672",Ohne Marker,41.708332,42.393618,56.905000
2,"10.04.2025 04:45:41,672","10.04.2025 04:47:41,672",Fremd,48.489944,49.266925,59.359994
3,"10.04.2025 04:50:39,672","10.04.2025 04:53:03,672",Fremd,47.296684,48.111554,59.961784
4,"10.04.2025 04:56:58,672","10.04.2025 04:57:31,672",Fremd,42.472237,43.495624,58.061701
5,"10.04.2025 04:57:58,672","10.04.2025 04:59:12,672",Fremd,43.006026,43.813803,59.999194
6,"10.04.2025 05:04:51,672","10.04.2025 05:05:45,672",Fremd,45.239153,45.841064,58.023768
7,"10.04.2025 05:08:26,672","10.04.2025 05:15:47,672",Silo,43.407683,44.241752,61.038871


In [63]:
summarkerdf

,marker,LAeq,LCeq,LAFmax
0,Fremd,45.918564,59.168156,46.702736
1,Gesamt,43.680663,58.734951,44.440906
2,Ohne Marker,41.708332,56.905000,42.393618
3,Silo,43.407683,61.038871,44.241752
